In [14]:
import pandas as pd
import torch
from pathlib import Path

In [15]:
carpeta = Path("../../data/traducidas")

In [16]:
df_es = pd.read_csv(carpeta / "resenas_espanol_trad.csv")
df_pt = pd.read_csv(carpeta / "resenas_portugues_trad.csv")

In [17]:
# Tamaño 
print("Dataset español:", df_es.shape)
print("Dataset portugués:", df_pt.shape)

Dataset español: (6632, 4)
Dataset portugués: (6632, 4)


In [18]:
# Muestra
muestra_es = df_es.sample(frac=0.10, random_state=42).copy()
muestra_pt = df_pt.sample(frac=0.10, random_state=42).copy()

In [19]:
# Tamaño de la muestra
print("Muestra español:", len(muestra_es))
print("Muestra portugués:", len(muestra_pt))

Muestra español: 663
Muestra portugués: 663


In [20]:
# Muestra portugués
muestra_pt

,Rating,Date,Review Text,Review Text_pt-BR
4554,5,6/12/2018,Best Stromboli and cannoli. Excellent customer...,Melhor Stromboli e cannoli. Ótimo atendimento ...
4864,5,3/17/2022,"I loved this ice cream, just the smell of this...","Eu adorei este sorvete, o aroma deste lugar é ..."
5099,1,10/7/2018,Staff was very rude! Me and my husband waited ...,A equipe foi muito mal-educada! Eu e meu marid...
4730,5,4/21/2019,I've always wanted to try this place and belie...,"Eu sempre quis experimentar este lugar e, acre..."
5998,5,3/25/2018,This place is great for late late night sweet ...,Este lugar é perfeito para aproveitar doces e ...
...,...,...,...,...
486,3,7/2/2021,Horrible customer service by Ingrid. Ordered s...,Atendimento terrível por parte de Ingrid. Pedi...
4684,5,12/7/2016,"Really, the best ice cream you can find in Veg...","De verdade, o melhor sorvete que você pode enc..."
5922,3,1/22/2017,I got the infamous cereal milk soft serve with...,Experimentei o famoso sorvete de leite de cere...
2963,2,1/15/2017,"Okay, so two weeks later almost all their cook...","Certo, então, duas semanas depois, quase todas..."


In [21]:
# Muestra español
muestra_es

,Rating,Date,Review Text,Review Text_es
4554,5,4/20/2009,The #1 best thing about Jarlings is the speedy...,"Lo mejor de Jarlings, sin duda, es su servicio..."
4864,4,3/27/2015,From reading other reviews (and my sister's pe...,Después de leer otras reseñas (y la experienci...
5099,3,4/4/2022,The doughnuts are ok but not for the price 15....,"Las donas están bien, pero no a este precio. 1..."
4730,4,3/1/2016,This place was pretty darn awesome. They put y...,Este lugar era realmente increíble. Te sirven ...
5998,2,6/5/2017,Cashiers il est mauvais. They are sarcastic sh...,El servicio es pésimo. Los empleados son sarcá...
...,...,...,...,...
486,3,6/21/2018,We were so excited to try the new flavors of t...,Estábamos muy emocionados de probar los nuevos...
4684,1,2/28/2016,Ice cream is over priced and they advertise ma...,"El helado es caro y, aunque publicitan macaron..."
5922,3,1/9/2019,Stopped in for a pastry and coffee. Wanted to ...,Fui a tomar un pastel y un café. Quería probar...
2963,1,12/26/2013,"OK, this is my first visit here and to be hone...","De acuerdo, esta es mi primera visita aquí, y ..."


In [ ]:
# Hugging Face
from huggingface_hub import whoami
print("Sesión de Hugging Face:", whoami()["name"])

Sesión de Hugging Face: Belu05


In [23]:
# COMET-Kiwi: carga del modelo
from comet import download_model, load_from_checkpoint

modelo_kiwi = load_from_checkpoint(download_model("Unbabel/wmt22-cometkiwi-da"))

c:\Users\belen\reviews-aa\.venv\Lib\site-packages\torchmetrics\utilities\imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]c:\Users\belen\reviews-aa\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\belen\.cache\huggingface\hub\models--Unbabel--wmt22-cometkiwi-da. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggin

In [24]:
# COMET-Kiwi
def calcular_kiwi(df, columna_traduccion):
    datos = []
    for original, traduccion in zip(df["Review Text"].astype(str), df[columna_traduccion].astype(str)):
        datos.append({"src": original, "mt": traduccion})

    salida = modelo_kiwi.predict(datos, batch_size=8, gpus=1 if torch.cuda.is_available() else 0)
    return salida.scores

In [25]:
# Prueba COMET-Kiwi: reseña en inglés, distintas traducciones al español
prueba = pd.DataFrame({
    "Review Text": ["The ice cream was delicious, but the service was very slow."] * 4,
    "traduccion": [
        "El helado estaba delicioso, pero la atención fue muy lenta.",        # correcta
        "El helado estaba rico, aunque el servicio fue realmente lento.",     # paráfrasis
        "El helado estaba delicioso y la atención fue muy rápida.",           # sentido opuesto
        "Ayer fui al parque con mi perro.",                                   # nada que ver
    ],
})

prueba["COMET_Kiwi"] = calcular_kiwi(prueba, "traduccion")
prueba[["traduccion", "COMET_Kiwi"]].round(3)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
c:\Users\belen\reviews-aa\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  1.57it/s]


,traduccion,COMET_Kiwi
0,"El helado estaba delicioso, pero la atención f...",0.881
1,"El helado estaba rico, aunque el servicio fue ...",0.884
2,El helado estaba delicioso y la atención fue m...,0.742
3,Ayer fui al parque con mi perro.,0.367


In [26]:
# COMET-Kiwi español
muestra_es["COMET_Kiwi"] = calcular_kiwi(muestra_es, "Review Text_es")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
c:\Users\belen\reviews-aa\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 83/83 [08:01<00:00,  5.81s/it]


In [27]:
# COMET-Kiwi portugués
muestra_pt["COMET_Kiwi"] = calcular_kiwi(muestra_pt, "Review Text_pt-BR")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
c:\Users\belen\reviews-aa\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 83/83 [07:48<00:00,  5.65s/it]


In [28]:
# Resultados COMET-Kiwi
print("COMET-Kiwi español:", round(muestra_es["COMET_Kiwi"].mean(), 4))
print("COMET-Kiwi portugués:", round(muestra_pt["COMET_Kiwi"].mean(), 4))

COMET-Kiwi español: 0.7984
COMET-Kiwi portugués: 0.7858


In [29]:
# Revisión español: las 10 peores traducciones
pd.set_option("display.max_colwidth", None)
muestra_es.nsmallest(10, "COMET_Kiwi")[["Review Text", "Review Text_es", "COMET_Kiwi"]]

,Review Text,Review Text_es,COMET_Kiwi
3149,"I used to sell Honey on the black market, and I got a good share of the profits (that was my Honey Cut), but I got sick of it (I was in a Honey Rut), so I opened a Crazy Golf course (named Honey Putt). One of my customers became the love of my life... we had nicknames for each other (she was Honey Butt) (I was Honey Nuts), and all was going well until we went bankrupt (Honey Kaput). So we started making adult movies together, and opened our own production company (Honey Smut), and it went so well that we were able to use the earnings to tour the country in search of the best Honey Pecan Ice Cream imaginable. And we found it. And it's in Cleveland. (drops mic) What to get: Honey Pecan Ice Cream Why: Because Honey Butt agrees, it's the best in the country Your dog won't like this ice cream, unless it's a Pecanese. If you close you're eyes, I'll hide your ice cream and we can see if you find it. Count to 40! No Pecan! The funny thing is, I don't even like Honey Pecan ice cream, mainly because I'm not crazy about nuts, but this thing is so good that I don't care - and you won't either. It's always busy here and the Honey Pecan is the reason why. There are several other locations in the area too, but this one is my favourite because it seems like all the cute customers hang out here, not to mention the girl that served me was a total flirt and while I am completely immune to such things... OK fine I'm not immune at all and I loved it. I need to pee and the only thing around is a can. I guess now it's a Peecan. Cleveland has a lot of wonderful places to eat main meals and snacks, but when it comes to dessert treats, you come here for ice cream, and then you go to Biagio's at 4am and get hot-out-of-the-fryer Crulers. And if the cute people here aren't enough for you and you're looking more for Playboy types, go to ""Bunny Hut."" If you want to date a comedian (like myself) try ""Funny Hut."" If Cleveland is too dreary for you, try the sister location in San Diego, ""Sunny Hut."" Or the Bank attached to it ""Money Hut."" Or the 2 places named after my jokes, ""Crummy Hut"" and ""Corny Hut."" And if all this has got you a little worked up and hot under the collar, then I'll meet you at ""Horny Hut.""","Antes vendía miel en el mercado negro, y obtenía una buena parte de las ganancias (esa era mi ""porción de miel""), pero me cansé de eso (estaba atrapado en una rutina), así que abrí un campo de mini-golf (llamado ""Honey Putt""). Uno de mis clientes se convirtió en el amor de mi vida... teníamos apodos para cada uno (ella era ""Honey Butt"", yo era ""Honey Nuts""), y todo iba bien hasta que nos quedamos sin dinero (Honey Kaput). Así que empezamos a hacer películas para adultos juntos, y abrimos nuestra propia productora (Honey Smut), y fue tan exitoso que pudimos usar las ganancias para hacer un tour por todo el país en busca del helado de avellanas y miel más increíble imaginable. Y lo encontramos. Y está en Cleveland. (suelta el micrófono) ¿Qué comprar: Helado de avellanas y miel? ¿Por qué? Porque ""Honey Butt"" está de acuerdo, es el mejor del país. A tu perro no le gustará este helado, a menos que sea de avellanas. Si cierras los ojos, te esconderé tu helado y veremos si lo encuentras. ¡Cuenta hasta 40! ¡No de avellanas! Lo gracioso es que yo tampoco me gusta el helado de avellanas, principalmente porque no me gustan las nueces, pero este es tan bueno que no me importa... y tampoco a ti. Siempre hay mucha gente aquí, y el helado de avellanas es la razón. También hay varios otros lugares en la zona, pero este es mi favorito porque parece que todos los clientes simpáticos se reúnen aquí, además, la chica que me atendió era una verdadera coqueta, y aunque estoy completamente inmune a esas cosas... OK, bueno, no estoy inmune en absoluto y me encantó. Necesito orinar, y lo único que hay alrededor es una lata. Supongo que ahora es de avellanas. Cleveland tiene muchos lugares maravillosos para c

In [30]:
# Revisión portugués: las 10 peores traducciones
muestra_pt.nsmallest(10, "COMET_Kiwi")[["Review Text", "Review Text_pt-BR", "COMET_Kiwi"]]

Review Text  \
2972                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              Let's get our sprinkle and sparkle on. Cream cheese frosting. Can you taste your favorite flavor, long for that soft fluffy sweetness. Cupcakes and ice cream in one place all together. Cupcakes will one day be the best way to celebrate our difficult differences. Decisions will be decided. Those that are in power must realize that the cylindrical shaped cups of love are way to powerful to resist. These treasured delights are the moments that are full of such harmony. Happiness can be achieved over coffee, tea, and cupcakes; period. The human race just needs to come to this realization before it's too late. So many delicious flavors and varieties could be achieved. When will they learn. Let's start the lesson without those individuals. We're talking cupcakes today, name the flavor and it's here. Deliciousness in a cylindrical culinary crush. Just walk through the door and let's start the process of ordering. The cupcakes are baked fresh here on sight each and every morning. The crew that calls this "work" knows that their company supports so many great causes and their ultimate happiness as equal human beings in all walks of life. Last month, June 2019 was Pride month. The whole month Sprinkles had a special cupcake just for this special occasion. They offered by special order an eight tier rainbow cake done just for pride month too. Companies that dedicate this much support to special causes are trying to make the world a better place. It's July 2019, Sprinkles is celebrating America's birthday with cupcakes that are red, white, and blue. They should be called Freedom cakes, but the causes that the company celebrates is authentic and geared towards many choices for all individuals to enjoy. The cupcakes always deliver here, never have I had a "bad" cupcake from any Sprinkles anywhere. This particular location is located on the board walk near the Linq Ferris wheel. A prime local right next to In and Out, by the large fountain in the court yard by the Flamingo Hotel and Casino. It's not easily allowable accessibility by car, but their is ample parking out by the Linq hotel. Unfortunately the parking for the Linq is no longer complimentary. The Flamingo Hotels parking garage is your best bet which is free. Now using the Valet at the Flamingo is costly. Either way you make your way to this Sprinkles it's worth the cost. Make a day out of it. Come for In and Out yummy burgers, then have cupcakes from Sprinkles for dessert. Come love one another, hand and hand, spread your love and beauty around. Embrace what it means to be accepted, no matter what your race, gender, sex, or other disposition in li